In [ ]:
import pandas as pd

import matplotlib.pyplot as plt

import numpy as np

In [ ]:
MAX_SEQ_LEN = 4096

MAX_TIME = 365

K = 5

TEST_SIZE = 0.2

FACTOR = 1/30

In [ ]:
def isEligible(x:pd.Series) -> int:
    if 'NFALG' in x or 'NFGEH' in x or 'NFVMI+' in x or 'NFVAL+' in x:
        return 1
    else:
        return 0

# Raggruppare per paziente, ordinare per data e filtrare in base alla condizione
def filter_positive_samples(group: pd.Series):
    # Se uno dei campioni è positivo, mantieni tutti i campioni precedenti
    if 1 in group['label'].values:
        # Troviamo l'indice del primo campione positivo
        first_positive_index = group[group['label'] == 1].index[0]
        # Mantieni tutti i campioni fino a questo indice incluso
        return group.loc[:first_positive_index]
    
    else:
        # Se non ci sono campioni positivi, mantieni tutti
        return group

# Average mean of the embeddings
def avg_mean_emb(group: pd.DataFrame) -> pd.DataFrame:
    numeric_data = group.select_dtypes(include=np.float32)
    
    delta = group['delta']
    
    weights = np.exp(-delta*FACTOR)
    
    weights = weights / weights.sum()
    
    weighted_avg = (numeric_data.T * weights).T.sum()
    
    output = group[['studyId_0831', 'recent_date', 'text', 'label']].iloc[0]
    
    output = pd.concat([output, weighted_avg])
    
    return output

## 1. Pre-processing of the patients agenda codes

In [ ]:
df_afsprak = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805\afspraken_20240805.sas7bdat',
    encoding='latin1'
)

df_afsprak['studyId_0831'] = df_afsprak['studyId_0831'].astype('uint64')

df_afsprak = df_afsprak.filter(items=['studyId_0831', 'index_AfspCode', 'period_start'])

df_afsprak = df_afsprak.sort_values(by=['studyId_0831', 'period_start'])

df_afsprak = df_afsprak.rename(columns={'period_start': 'date'})

# Rimuovo l'orario dalla data
df_afsprak['date'] = df_afsprak['date'].dt.normalize()

df_afsprak = df_afsprak.reset_index(drop=True)

df_afsprak

### 1.1 Viene applicata la funzione `is_Eligible` per assegnare la label ad ogni paziente

In [ ]:
df_afsprak = df_afsprak.drop_duplicates()

# Applico la funzione "isEligible" alla colonna degli agendacodes
df_afsprak['label'] = df_afsprak['index_AfspCode'].apply(isEligible)

df_afsprak

### 1.2 Dataframe filtering
For each patient, the sample corresponding to the first insertion date into the CVRM group is retained, along with any previous samples if available. If the patient does not belong to the CVRM group, the most recent appointment date is used. To consider the most relevant patient history, all samples with a time difference greater than `MAX_TIME` from the most recent date are discarded.

In [ ]:
# Applicare la funzione di filtraggio al gruppo di pazienti
df_afsprak_filt = df_afsprak.groupby('studyId_0831')[['studyId_0831', 'date', 'label']].apply(filter_positive_samples).reset_index(drop=True)

df_afsprak_filt

In [ ]:
# Calcolo la data più recente (ultima data) per ogni paziente
df_afsprak_filt['last_date'] = df_afsprak_filt.groupby('studyId_0831')['date'].transform('max')

# Calcolo la differenza in giorni tra l'ultima data e tutte le altre date per ogni paziente
df_afsprak_filt['delta'] = (df_afsprak_filt['last_date'] - df_afsprak_filt['date']).dt.days

df_afsprak_filt

In [ ]:
# Filtro i campioni che hanno una differenza maggiore di MAX_TIME
df_afsprak_recents = df_afsprak_filt[df_afsprak_filt['delta'] <= MAX_TIME].reset_index(drop=True)

df_afsprak_recents = df_afsprak_recents.drop(columns=['last_date', 'delta'])

# Rimuovo i campioni con entrate precedenti alla prima data utile
df_afsprak_recents = df_afsprak_recents.drop_duplicates(
    subset = 'studyId_0831',
    keep = 'last'
)

df_afsprak_recents = df_afsprak_recents.rename(columns={'date': 'recent_date'})

df_afsprak_recents

## 2. Aggiunta dei consulti dei pazienti al dataframe delle labels

In [ ]:
df_consults = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")

df_consults['studyId_0831'] = df_consults['studyId_0831'].astype('uint64')

# Rimuovo l'orario dalla data
df_consults['date'] = df_consults['date'].dt.normalize()

df_consults = df_consults.sort_values(by=['studyId_0831', 'date']).reset_index(drop=True)

df_consults

In [ ]:
num_unique_not_in_both = len(set(df_consults['studyId_0831']) ^ set(df_afsprak['studyId_0831']))

# Pazienti mancanti nel consults
missing_in_consults = list(set(df_afsprak['studyId_0831']) - set(df_consults['studyId_0831']))

print(f'Missing patients from the consults dataset ({num_unique_not_in_both}): \n{missing_in_consults}')

In [ ]:
df_afsprak_consults = pd.merge(
    left = df_consults,
    right = df_afsprak_recents,
    on = 'studyId_0831',
    how = 'right'
)

df_afs_cons_med = df_afsprak_consults.dropna()

df_afsprak_consults

In [ ]:
# Calcolo la differenza in giorni tra l'ultima data e tutte le altre date per ogni paziente
df_afsprak_consults['delta'] = (df_afsprak_consults['recent_date'] - df_afsprak_consults['date']).dt.days

df_afsprak_consults

In [ ]:
df_afsprak_consults_filt = df_afsprak_consults[
    (df_afsprak_consults['delta'] >= 0) & (df_afsprak_consults['delta'] <= MAX_TIME)
]

df_afsprak_consults_filt = df_afsprak_consults_filt.drop_duplicates(
    subset = ['studyId_0831', 'date', 'text']
)

df_afsprak_consults_filt

In [ ]:
# Raggruppo per 'studyId_0831', seleziono la data più recente e unisco i testi
df_afsprak_consults_aggregated = df_afsprak_consults_filt.groupby(by=['studyId_0831', 'recent_date'])\
    .agg({'text': '\n'.join, 'label': 'first'}).reset_index()

df_afsprak_consults_aggregated

In [ ]:
# Calcolo la lunghezza di ogni testo
num_words = df_afsprak_consults_aggregated['text'].str.split().str.len()

# Traccio l'istogramma della colonna 'num_words'
num_words.plot(kind='hist', bins=20, edgecolor='black')

# Aggiungo il titolo e le etichette
plt.title('Frequency of the consults length')
plt.xlabel('Number of words')
plt.ylabel('Frequency')

# Mostrare il grafico
plt.show()

# Filtro i campioni con un numero di parole inferiore al limite
campioni_sotto_limite = len(
    df_afsprak_consults_aggregated[num_words < MAX_SEQ_LEN]
)

# Calcolo la percentuale di campioni sotto il limite
percentuale = (campioni_sotto_limite / len(num_words)) * 100

# Visualizzo il risultato
print(f"The percentage of samples with fewer than {MAX_SEQ_LEN} words is: {percentuale:.2f}%")

## 3. Aggiunta degli emdeddings degli ATC codes al dataframe precedente

In [ ]:
df_med_emb = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_medicatie_embeddings.parquet')

# Rimuovo l'orario dalla data
df_med_emb['date'] = df_med_emb['date'].dt.normalize()

df_med_emb = df_med_emb.sort_values(by=['studyId_0831', 'date'])

df_med_emb = df_med_emb.drop(columns='code5_ATC_code')

df_med_emb

In [ ]:
num_unique_not_in_both = len(set(df_med_emb['studyId_0831']) ^ set(df_afsprak['studyId_0831']))

missing_in_med_emb = list(set(df_afsprak['studyId_0831']) - set(df_med_emb['studyId_0831']))

print(f'Missing patients from the consults dataset ({num_unique_not_in_both}): \n{missing_in_med_emb}')

In [ ]:
df_afs_cons_med = pd.merge(
    left = df_afsprak_consults_aggregated,
    right = df_med_emb,
    on = 'studyId_0831',
    how = 'left'
)

df_afs_cons_med = df_afs_cons_med.dropna()

df_afs_cons_med

In [ ]:
# Calcolo la differenza in giorni tra l'ultima data e tutte le altre date per ogni paziente
df_afs_cons_med['delta'] = (df_afs_cons_med['recent_date'] - df_afs_cons_med['date']).dt.days

df_afs_cons_med

In [ ]:
df_afs_cons_med_filt = df_afs_cons_med[
    (df_afs_cons_med['delta'] >= 0) & (df_afs_cons_med['delta'] <= MAX_TIME)
]

df_afs_cons_med_filt

In [ ]:
df_afs_cons_med_avg = df_afs_cons_med_filt.groupby('studyId_0831').apply(avg_mean_emb)

df_afs_cons_med_avg = df_afs_cons_med_avg.reset_index(drop=True)

df_afs_cons_med_avg

In [ ]:
df_afs_cons_med_avg['label'].hist(bins=2, edgecolor='black')
plt.xlabel('Valori')
plt.ylabel('Frequenza')
plt.title('Distribuzione dei valori nella colonna')
plt.grid(False)
plt.show()